# Dimension categories per abstract: Jev vs LLM (pilot)

`scripts/semantic_analysis/assess_dimension_categories.py` assigns the dimension categories per *cluster*, from an LLM narrative built on a sample of abstracts. This notebook tests whether the categories can be assigned per *abstract* instead, using [Jev](https://www.glean.com/blog/jev-zero-shot-classifier) (typed decisions: context + fixed questions/options → choices and probabilities).

1. Draw a small sample of research abstracts (one per randomly chosen cluster)
2. Write the template for **my own annotation** of a 5-abstract subset (fill it in *before* looking at sections 3–4)
3. LLM baseline: all dimensions in a single call per abstract (gpt-4o and gpt-4o-mini)
4. Jev (TypeSafe API): all questions for an abstract in a single call, in three formulations
5. Speed and projected cost for the full corpus
6. Agreement Jev vs LLM
7. Agreement with my annotation

The category lists are those of `assess_dimension_categories.py`. Each category is a binary yes/no decision, as in the cluster-level analysis. Model outputs are cached in `OUTPUT_PATH`, so re-running the notebook does not repeat API calls.

Requires `OPENAI_API_KEY` and `JEV_API_KEY` (a TypeSafe key) in `.env`.

In [1]:
import os
import json
import time
import tomllib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
sns.set_context('notebook')

BASEPATH = os.environ.get('BASEPATH_V2', '/media/mario/HDD/Data/NeuroScape_v2')
csv_path = os.path.join(BASEPATH, 'Public', 'Data', 'CSV')
OUTPUT_PATH = os.path.join(BASEPATH, 'Internal', 'Checkpoints', 'abstract_dimensions_pilot')
os.makedirs(OUTPUT_PATH, exist_ok=True)

with open('../config/analysis/semantic.toml', 'rb') as f:
    configurations = tomllib.load(f)

N_SAMPLE = 30   # abstracts for the Jev vs LLM comparison
N_HUMAN = 5     # subset of those annotated by hand
SEED = 2026

## Dimensions and categories

In [2]:
DIMENSION_CATEGORIES = {
    'Appliedness': [
        'Fundamental', 'Translational', 'Clinical', 'Legal', 'Economic',
        'Method Development', 'Technological Exploitation'
    ],
    'Modality': [
        'Auditory', 'Visual', 'Olfactory', 'Gustatory', 'Somatosensory',
        'Multimodal', 'Visuomotor', 'Sensorimotor', 'Motor', 'Emotional',
        'Behavioral', 'Cognitive'
    ],
    'Spatial Scale': [
        'Molecular', 'Cellular', 'Circuit', 'Region', 'Systems', 'Whole-brain'
    ],
    'Temporal Scale': [
        'Microsecond', 'Millisecond', 'Second', 'Minute', 'Hour', 'Day',
        'Week', 'Month', 'Year', 'Lifetime'
    ],
    'Cognitive Complexity':
    ['Low-level Sensory', 'Low-level Motor', 'Mid-level', 'High-level'],
    'Species': [
        'Human', 'Non-human primates', 'Rodents', 'Mammals', 'Birds', 'Fish',
        'Amphibians', 'Invertebrates', 'Cell cultures', 'Other'
    ],
    'Theory Engagement': ['Data-driven', 'Hypothesis-driven'],
    'Theory Scope': [
        'Overarching Framework', 'Domain Framework',
        'Disease-specific Framework', 'Micro Theory'
    ],
    'Methodological Approach': [
        'Experimental', 'Observational', 'Computational', 'Theoretical',
        'Meta-analytic'
    ],
    'Interdisciplinarity': ['Low', 'Medium', 'High', 'Very High']
}

# Descriptions taken from the prompt of assess_dimension_categories.py
DIMENSION_DESCRIPTIONS = {
    'Appliedness': 'The extent to which the research is basic science (fundamental) or applied in one of several ways.',
    'Modality': 'The sensory and/or motor modality under investigation.',
    'Spatial Scale': 'The spatial scale of the research, from microscale to macroscale.',
    'Temporal Scale': 'The temporal scale of the research, from microscale to macroscale.',
    'Cognitive Complexity': 'The level of cognitive complexity under investigation from low level (e.g., sensory processing, motor control) to high level (e.g., language, decision making, social cognition).',
    'Species': 'The species under investigation.',
    'Theory Engagement': 'The extent to which the research is theory-driven (hypothesis testing) or data-driven (exploratory, descriptive).',
    'Theory Scope': (
        'The scope of the theory under investigation. '
        'Overarching Framework: a broad theoretical approach that aims to explain fundamental principles of brain function across multiple cognitive and neural domains. '
        'Domain Framework: an integrative theory focusing on a specific subfield, offering cohesive principles for that area. '
        'Disease-specific Framework: a theory that details the neural causes, mechanisms, and manifestations of a particular neurological or psychiatric condition. '
        'Micro Theory: a narrowly scoped, mechanistic account or model that explains one specific process or phenomenon in the brain.'),
    'Methodological Approach': (
        'The methodological approach used in the research. '
        'Experimental: researchers deliberately manipulate one or more variables under controlled conditions to test for causal effects. '
        'Observational (Correlational / Descriptive): variables are measured in naturally occurring settings without active intervention. '
        'Computational / coding: mathematical, algorithmic, or simulation-based models are constructed or tested to predict, explain, or interpret empirical data or biological processes. '
        'Theoretical / Conceptual: conceptual frameworks and theories are developed, refined, or critiqued without new empirical data or simulations. '
        'Meta-Analytic / Systematic Review: existing primary studies are synthesized and reanalyzed systematically.'),
    'Interdisciplinarity': (
        'The extent to which the research is interdisciplinary, combining methods and concepts from multiple fields. '
        'From low (confined to a single discipline) to very high (incorporating multiple disciplines in a transdisciplinary manner). '
        'Multidisciplinary: multiple disciplines study the same problem in parallel with little cross-integration. '
        'Interdisciplinary: theories, methods, or data from different disciplines are integrated into shared frameworks. '
        'Transdisciplinary: collaboration goes beyond academic boundaries, or disciplines merge so completely that new fields emerge.')
}

KEYS = ['Pmid', 'Dimension', 'Category']
N_DECISIONS = sum(len(c) for c in DIMENSION_CATEGORIES.values())
print(f'{len(DIMENSION_CATEGORIES)} dimensions, {N_DECISIONS} binary decisions per abstract')

10 dimensions, 64 binary decisions per abstract


## 1. Sample

One research article per randomly chosen cluster, so that the sample covers a broad range of topics. The first `N_HUMAN` abstracts form the hand-annotated subset.

In [3]:
article_df = pd.read_csv(os.path.join(csv_path, 'neuroscience_articles_1999-2025.csv'),
                         low_memory=False)
research_df = article_df[(article_df['Type'] == 'Research') & article_df['Abstract'].notna()]

rng = np.random.default_rng(SEED)
clusters = rng.choice(research_df['Cluster ID'].unique(), size=N_SAMPLE, replace=False)
sample_df = (research_df[research_df['Cluster ID'].isin(clusters)]
             .groupby('Cluster ID').sample(n=1, random_state=SEED)
             .set_index('Cluster ID').loc[clusters].reset_index()
             [['Pmid', 'Cluster ID', 'Year', 'Title', 'Abstract']])
human_pmids = sample_df['Pmid'].iloc[:N_HUMAN].tolist()

print(f'{len(sample_df)} abstracts, median length {sample_df["Abstract"].str.split().str.len().median():.0f} words')
sample_df[['Pmid', 'Cluster ID', 'Year', 'Title']]

30 abstracts, median length 228 words


,Pmid,Cluster ID,Year,Title
0,18439802,120,2008,Iterative principles of recognition in probabi...
1,20826314,155,2010,Neuronal activity drives localized blood-brain...
2,15619461,52,2005,S100B and NSE serum levels in patients with Pa...
3,26859852,143,2016,Altered hemispheric lateralization of white ma...
4,30519212,76,2018,White Matter Changes Along the Electrode Lead ...
5,33314354,3,2021,Novel factor in olfactory ensheathing cell-ast...
6,18387672,37,2008,Neurotoxicity of MPTP to migrating neuroblasts...
7,25977323,104,2015,Alleviating simulator sickness with galvanic c...
8,29660375,123,2018,The secretome of adipose-derived mesenchymal s...
9,21985904,133,2012,In vivo evaluation of retinal and callosal pro...


In [4]:
def article_context(row):
    # The text every rater (LLM, Jev, me) sees for an article
    return f"Title: {row['Title']}\n\nAbstract: {row['Abstract']}"


contexts = {row['Pmid']: article_context(row) for _, row in sample_df.iterrows()}

## 2. My annotation (do this first)

The cell below writes `human_annotation.csv` (never overwritten once it exists): one row per abstract and dimension. In `Selected Categories`, list the categories that apply, separated by `;`, exactly as spelled in `Allowed Categories` (case does not matter). Leave it empty if none apply.

To stay blind, annotate before running or reading sections 3–4.

In [5]:
human_file = os.path.join(OUTPUT_PATH, 'human_annotation.csv')

if not os.path.exists(human_file):
    pd.DataFrame([
        {'Pmid': pmid, 'Dimension': dimension,
         'Allowed Categories': '; '.join(categories), 'Selected Categories': ''}
        for pmid in human_pmids
        for dimension, categories in DIMENSION_CATEGORIES.items()
    ]).to_csv(human_file, index=False)
    print(f'Template written to {human_file}')
else:
    print(f'{human_file} exists, leaving it untouched')

for pmid in human_pmids:
    print(f'=== PMID {pmid} ===\n{contexts[pmid]}\n')

/media/mario/HDD/Data/NeuroScape_v2/Internal/Checkpoints/abstract_dimensions_pilot/human_annotation.csv exists, leaving it untouched
=== PMID 18439802 ===
Title: Iterative principles of recognition in probabilistic neural networks.

Abstract: When considering the probabilistic approach to neural networks in the framework of statistical pattern recognition we assume approximation of class-conditional probability distributions by finite mixtures of product components. The mixture components can be interpreted as probabilistic neurons in neurophysiological terms and, in this respect, the fixed probabilistic description contradicts the well known short-term dynamic properties of biological neurons. By introducing iterative schemes of recognition we show that some parameters of probabilistic neural networks can be "released" for the sake of dynamic processes without disturbing the statistically correct decision making. In particular, we can iteratively adapt the mixture component weights or

## 3. LLM: all dimensions in one call

Same prompt for two models: the model of the cluster-level analysis (`config/analysis/semantic.toml`) and the much cheaper gpt-4o-mini. Temperature 0, since this is classification. Token usage is recorded for the cost estimate.

In [6]:
from langchain_openai import ChatOpenAI
import openai
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
openai.api_key = os.environ["OPENAI_API_KEY"]

LLM_MODELS = [configurations['llm']['model_name'], 'gpt-4o-mini']
# USD per 1M input and output tokens; check against the current OpenAI pricing
LLM_PRICES = {'gpt-4o-2024-08-06': (2.50, 10.00), 'gpt-4o-mini': (0.15, 0.60)}

dimension_block = '\n'.join(
    f'{i}. {dimension}: {DIMENSION_DESCRIPTIONS[dimension]}\n   Categories: {", ".join(categories)}'
    for i, (dimension, categories) in enumerate(DIMENSION_CATEGORIES.items(), start=1))

json_skeleton = json.dumps(
    {dimension: {category: 'yes / no' for category in categories}
     for dimension, categories in DIMENSION_CATEGORIES.items()}, indent=1)

LLM_PROMPT = '''You are an expert in neuroscience.
You are given the title and abstract of a single neuroscientific article. Assess the article along the following dimensions:

{dimensions}

For every category of every dimension, indicate with "yes" or "no" whether the article falls within that category. Several categories of a dimension can apply, or none.

Instructions:
- Base every decision on the title and abstract only. Do not make external assumptions.
- Use exactly the dimension and category names given above.
- Answer only with JSON of the following structure, with "yes" or "no" for each category:
{skeleton}

{context}'''

In [7]:
def parse_llm_output(output):
    # Long-format records, or None if the output is incomplete or invalid
    conversion = {'yes': True, 'no': False}
    records = []
    try:
        for dimension, categories in DIMENSION_CATEGORIES.items():
            for category in categories:
                value = str(output[dimension][category]).strip().lower()
                records.append({'Dimension': dimension, 'Category': category,
                                'Value': conversion[value]})
    except (KeyError, TypeError):
        return None
    return records


def assess_with_llm(pmid, llm, retries=configurations['llm']['retries']):
    # Returns the records plus timing and token usage
    prompt = LLM_PROMPT.format(dimensions=dimension_block, skeleton=json_skeleton,
                               context=contexts[pmid])
    start = time.perf_counter()
    input_tokens = output_tokens = 0
    for attempt in range(1, retries + 1):
        response = llm.invoke(prompt)
        input_tokens += response.usage_metadata['input_tokens']
        output_tokens += response.usage_metadata['output_tokens']
        try:
            records = parse_llm_output(json.loads(response.content))
        except json.JSONDecodeError:
            records = None
        if records:
            break
    else:
        raise RuntimeError(f'No valid LLM output for {pmid} after {retries} attempts')
    timing = {'Pmid': pmid, 'Seconds': time.perf_counter() - start, 'Calls': attempt,
              'Input Tokens': input_tokens, 'Output Tokens': output_tokens}
    return [{'Pmid': pmid, **r} for r in records], timing

In [ ]:
def run_cached(name, assess, pmids):
    # Run `assess` on the pmids not yet in the cache, appending to <name>.csv and <name>_timing.csv
    result_file = os.path.join(OUTPUT_PATH, f'{name}.csv')
    timing_file = os.path.join(OUTPUT_PATH, f'{name}_timing.csv')
    done = set(pd.read_csv(timing_file)['Pmid']) if os.path.exists(timing_file) else set()

    for pmid in tqdm([p for p in pmids if p not in done], desc=name):
        records, timing = assess(pmid)
        for df, file in [(pd.DataFrame(records), result_file), (pd.DataFrame([timing]), timing_file)]:
            df.to_csv(file, mode='a', header=not os.path.exists(file), index=False)

    return pd.read_csv(result_file), pd.read_csv(timing_file)


# Per method: long-format results, timing, and USD per 1M input / output tokens
results = {}
for model in LLM_MODELS:
    llm = ChatOpenAI(model_name=model, temperature=0,
                     model_kwargs={'response_format': {'type': 'json_object'}})
    df, timing = run_cached(f'llm_{model}', lambda pmid: assess_with_llm(pmid, llm), sample_df['Pmid'])
    results[f'LLM {model}'] = {'df': df, 'timing': timing, 'price': LLM_PRICES[model]}

REFERENCE = f'LLM {LLM_MODELS[0]}'
pd.concat({name: r['timing'][['Seconds', 'Input Tokens', 'Output Tokens']].median()
           for name, r in results.items()}, axis=1)

## 4. Jev

Via the TypeSafe API (`POST /v1/systemone`). There is no limit on the number of questions per request, so each abstract takes a single call. Three formulations:

- `'packed'`: one `noul` (yes/no) question per category, giving P(yes); a category is selected if P(yes) ≥ `JEV_THRESHOLD`. Several or no categories per dimension.
- `'gated'`: as `'packed'`, plus one `choice` question per dimension on whether it applies at all (applies / not applicable / not stated). If it does not apply or is not stated, no category is selected; if it applies but no category reaches the threshold, the most likely one is selected.
- `'choice'`: one `choice` question per dimension among its categories plus "Not applicable" and "Not stated". Exactly one category, or none.

Billing is per input token (output is free); the state is billed once per call plus the instructions of each question.

In [ ]:
import requests

JEV_URL = os.environ.get('JEV_BASE_URL', 'https://api.typesafe.ai') + '/v1/systemone'
JEV_PROVIDER = 'typesafe' if 'typesafe' in JEV_URL else 'jevmodel'
JEV_MODEL = 'jev-1.13.0'  # pinned, so that thresholds stay valid
JEV_MODES = ['packed', 'gated', 'choice']
JEV_THRESHOLD = 0.5
JEV_MAX_QUESTIONS = None  # per call; no limit on TypeSafe, 8 on jevmodel.org
JEV_PRICES = {'typesafe': 0.042, 'jevmodel': 0.66}  # USD per 1M input tokens

APPLIES, NOT_APPLICABLE, NOT_STATED = 'Applies', 'Not applicable', 'Not stated'
ABSTENTIONS = {
    NOT_APPLICABLE: 'The dimension does not apply to this article.',
    NOT_STATED: 'The dimension applies, but the title and abstract do not say which category.'
}

jev_session = requests.Session()
jev_session.headers.update({'Authorization': f'Bearer {os.environ["JEV_API_KEY"]}',
                            'Content-Type': 'application/json'})


def jev_decide(context, questions, retries=5):
    '''
    Send one request to Jev.

    Parameters:
    - context: str, the title and abstract.
    - questions: list of dict with keys 'id' and 'question' (a question as in the API).
    - retries: int, the number of retries on rate limit or upstream errors.

    Returns:
    - answers: dict mapping question id to the answer of the API.
    - input_tokens: int, the billed input tokens.
    '''
    # Plain ids in the request, mapped back afterwards
    ids = {f'q{i}': q['id'] for i, q in enumerate(questions)}
    body = {'model': JEV_MODEL, 'state': context,
            'questions': {qid: q['question'] for qid, q in zip(ids, questions)}}
    for attempt in range(retries + 1):
        response = jev_session.post(JEV_URL, json=body, timeout=120)
        if response.status_code in (429, 502, 529) and attempt < retries:
            time.sleep(float(response.headers.get('Retry-After', 2 ** attempt)))
            continue
        response.raise_for_status()
        break
    output = response.json()
    answers = {ids[qid]: answer for qid, answer in output['answers'].items()}
    return answers, output['usage']['input_tokens']


def noul_questions(dimension):
    # One yes/no question per category of the dimension
    return [
        {'id': f'{dimension}|{category}',
         'question': {'type': 'noul',
                      'instructions': (f'Dimension "{dimension}": {DIMENSION_DESCRIPTIONS[dimension]} '
                                       f'Based on the title and abstract only, does the article fall within '
                                       f'the category "{category}" of this dimension?')}}
        for category in DIMENSION_CATEGORIES[dimension]
    ]


def gate_question(dimension):
    # Whether the dimension applies to the article and the abstract says enough to categorize it
    return {'id': f'{dimension}|gate',
            'question': {'type': 'choice',
                         'instructions': (f'Dimension "{dimension}": {DIMENSION_DESCRIPTIONS[dimension]} '
                                          f'Categories: {", ".join(DIMENSION_CATEGORIES[dimension])}. '
                                          f'Based on the title and abstract only, can the article be '
                                          f'placed in one or more of these categories?'),
                         'criteria': {APPLIES: 'The dimension applies and the title and abstract '
                                               'indicate one or more of the categories.',
                                      **ABSTENTIONS}}}


def choice_question(dimension):
    # A single category of the dimension, or an abstention
    return {'id': dimension,
            'question': {'type': 'choice',
                         'instructions': (f'Dimension "{dimension}": {DIMENSION_DESCRIPTIONS[dimension]} '
                                          f'Which category best describes the article, based on the '
                                          f'title and abstract only?'),
                         'criteria': {**{c: c for c in DIMENSION_CATEGORIES[dimension]}, **ABSTENTIONS}}}


def chunk(questions, size=JEV_MAX_QUESTIONS):
    size = size or len(questions)
    return [questions[i:i + size] for i in range(0, len(questions), size)]


def interpret_dimension(answers, dimension, mode):
    # Per category of the dimension: (selected, probability, dimension-level choice)
    categories = DIMENSION_CATEGORIES[dimension]
    if mode == 'choice':
        answer = answers[dimension]
        return {c: (answer['choice'] == c, answer['probabilities'][c], answer['choice'])
                for c in categories}

    noul = {c: answers[f'{dimension}|{c}']['noul'] for c in categories}
    selected = {c: p >= JEV_THRESHOLD for c, p in noul.items()}
    gate = None
    if mode == 'gated':
        gate = answers[f'{dimension}|gate']['choice']
        if gate != APPLIES:
            selected = dict.fromkeys(categories, False)
        elif not any(selected.values()):
            selected[max(noul, key=noul.get)] = True
    return {c: (selected[c], noul[c], gate) for c in categories}


def assess_with_jev(pmid, mode):
    dimensions = list(DIMENSION_CATEGORIES)
    if mode == 'packed':
        questions = [q for d in dimensions for q in noul_questions(d)]
    elif mode == 'gated':
        questions = [q for d in dimensions for q in noul_questions(d) + [gate_question(d)]]
    elif mode == 'choice':
        questions = [choice_question(d) for d in dimensions]
    else:
        raise ValueError(f'Unknown mode {mode}')

    start = time.perf_counter()
    answers, input_tokens = {}, 0
    for batch in chunk(questions):
        batch_answers, batch_tokens = jev_decide(contexts[pmid], batch)
        answers.update(batch_answers)
        input_tokens += batch_tokens
    timing = {'Pmid': pmid, 'Seconds': time.perf_counter() - start,
              'Calls': len(chunk(questions)), 'Mode': mode, 'Input Tokens': input_tokens}

    records = [
        {'Pmid': pmid, 'Dimension': d, 'Category': c, 'Value': value, 'Probability': p, 'Choice': choice}
        for d in dimensions
        for c, (value, p, choice) in interpret_dimension(answers, d, mode).items()
    ]
    return records, timing

In [ ]:
# Smoke test on one abstract before running the sample
for mode in JEV_MODES:
    records, timing = assess_with_jev(sample_df['Pmid'].iloc[0], mode)
    print(timing)
pd.DataFrame(records).head(12)

In [ ]:
for mode in JEV_MODES:
    df, timing = run_cached(f'jev_{mode}_{JEV_PROVIDER}', lambda pmid: assess_with_jev(pmid, mode),
                            sample_df['Pmid'])
    results[f'Jev {mode}'] = {'df': df, 'timing': timing, 'price': (JEV_PRICES[JEV_PROVIDER], 0)}

# Earlier run through the jevmodel.org reseller (8 questions per call), for speed and cost only
if os.path.exists(os.path.join(OUTPUT_PATH, 'jev_packed_jevmodel.csv')):
    df, timing = run_cached('jev_packed_jevmodel', None, [])
    jevmodel_run = {'df': df, 'timing': timing, 'price': (JEV_PRICES['jevmodel'], 0)}

pd.concat({name: r['timing'][['Seconds', 'Calls', 'Input Tokens']].median()
           for name, r in results.items() if name.startswith('Jev')}, axis=1)

## 5. Speed and projected cost

Wall time per abstract, measured sequentially. The projection for the full corpus assumes requests can be parallelised with `CONCURRENCY` workers; the actual rate limits (OpenAI tier, TypeSafe organisation limits in the console) may cap this. The jevmodel.org row is the earlier reseller run, for comparison.

In [ ]:
CONCURRENCY = 8
n_corpus = len(research_df)

speed_runs = {**results, 'Jev packed (jevmodel.org)': jevmodel_run} if 'jevmodel_run' in globals() else results


def usd_per_abstract(run):
    price_input, price_output = run['price']
    timing = run['timing']
    output_tokens = timing['Output Tokens'] if 'Output Tokens' in timing else 0
    return ((timing['Input Tokens'] * price_input + output_tokens * price_output) / 1e6).mean()


speed_df = pd.DataFrame({
    name: {'Seconds / abstract (median)': run['timing']['Seconds'].median(),
           'Calls / abstract': run['timing']['Calls'].mean(),
           'Input Tokens / abstract': run['timing']['Input Tokens'].mean(),
           'USD / abstract': usd_per_abstract(run)}
    for name, run in speed_runs.items()
}).T
speed_df[f'Corpus hours ({CONCURRENCY} workers)'] = (
    speed_df['Seconds / abstract (median)'] * n_corpus / CONCURRENCY / 3600)
speed_df['Corpus USD'] = speed_df['USD / abstract'] * n_corpus

print(f'{n_corpus:,} research articles in the corpus')
speed_df.round(4)

In [ ]:
timing_df = pd.concat([run['timing'].assign(Method=name) for name, run in speed_runs.items()])

fig, ax = plt.subplots(figsize=(8, 4))
sns.stripplot(data=timing_df, x='Method', y='Seconds', ax=ax, alpha=0.7)
ax.set_yscale('log')
ax.set_ylabel('Seconds per abstract')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
sns.despine()
plt.tight_layout()
plt.show()

## 6. Agreement between methods

Every method against the reference LLM (the model of the cluster-level analysis). Per dimension: raw agreement over all its binary decisions, Cohen's kappa over the same decisions pooled across categories (raw agreement alone is inflated because most categories are "no"), the share of abstracts for which the whole dimension matches, and the "yes" rate of each rater.

In [ ]:
def cohen_kappa(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    observed = np.mean(a == b)
    expected = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return np.nan if expected == 1 else (observed - expected) / (1 - expected)


def agreement(df_a, df_b, name_a, name_b):
    '''
    Agreement between two raters per dimension and overall.

    Parameters:
    - df_a, df_b: pd.DataFrame, long format with columns Pmid, Dimension, Category, Value.
    - name_a, name_b: str, the rater names.

    Returns:
    - summary: pd.DataFrame
    '''
    merged = df_a[KEYS + ['Value']].merge(df_b[KEYS + ['Value']], on=KEYS, suffixes=('_a', '_b'))
    merged['Match'] = merged['Value_a'] == merged['Value_b']

    def summarize(group):
        return pd.Series({
            'Agreement': group['Match'].mean(),
            'Kappa': cohen_kappa(group['Value_a'], group['Value_b']),
            'Dimension Match': group.groupby('Pmid')['Match'].all().mean(),
            f'Yes Rate {name_a}': group['Value_a'].mean(),
            f'Yes Rate {name_b}': group['Value_b'].mean(),
        })

    summary = merged.groupby('Dimension', sort=False).apply(summarize, include_groups=False)
    summary.loc['Overall'] = summarize(merged)
    summary['N Abstracts'] = merged['Pmid'].nunique()
    return summary


def kappa_table(runs, reference_df, reference_name):
    # Kappa per dimension (and overall) of each run against a reference
    return pd.DataFrame({name: agreement(run['df'], reference_df, name, reference_name)['Kappa']
                         for name, run in runs.items()})


others = {name: run for name, run in results.items() if name != REFERENCE}
kappa_table(others, results[REFERENCE]['df'], REFERENCE).round(2)

In [ ]:
def disagreements(df_a, df_b, name_a, name_b):
    # Categories ranked by how often the two raters disagree
    merged = df_a[KEYS + ['Value']].merge(df_b[KEYS + ['Value']], on=KEYS,
                                          suffixes=(f' {name_a}', f' {name_b}'))
    a, b = merged[f'Value {name_a}'], merged[f'Value {name_b}']
    merged[f'Only {name_a}'] = a & ~b
    merged[f'Only {name_b}'] = ~a & b
    return (merged.groupby(['Dimension', 'Category'])[[f'Only {name_a}', f'Only {name_b}']].sum()
            .assign(Total=lambda d: d.sum(axis=1)).sort_values('Total', ascending=False))


disagreements(results['Jev gated']['df'], results[REFERENCE]['df'], 'Jev', 'LLM').head(15)

For the `noul` formulations, the decision threshold on P(yes) can be varied. If Jev's default is systematically more or less liberal than the LLM, a different threshold might agree better. With 30 abstracts this is only indicative. (For `'gated'`, only the category questions are re-thresholded; the gate stays as answered.)

In [ ]:
def rethreshold(df, threshold):
    # Selections at another threshold; keeps the gate of the 'gated' formulation
    df = df.copy()
    gate_open = df['Choice'].isna() | (df['Choice'] == APPLIES)
    df['Value'] = gate_open & (df['Probability'] >= threshold)
    return df


thresholds = np.linspace(0.05, 0.95, 19)
fig, ax = plt.subplots(figsize=(5, 4))
for name in ['Jev packed', 'Jev gated']:
    kappas = [agreement(rethreshold(results[name]['df'], t), results[REFERENCE]['df'], 'Jev', 'LLM')
              .loc['Overall', 'Kappa'] for t in thresholds]
    ax.plot(thresholds, kappas, marker='o', label=name)
ax.set_xlabel('Threshold on Jev P(yes)')
ax.set_ylabel(f"Cohen's kappa with {REFERENCE}")
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## 7. Agreement with my annotation

On the `N_HUMAN` abstracts only, so treat the numbers as anecdotal; the disagreement tables are the more useful output. Where I answered "not applicable" or "don't know", the annotation has no category selected (all False).

In [ ]:
def load_human_annotation(file):
    # Long format with Value True for the selected categories; checks spelling
    annotation = pd.read_csv(file, keep_default_na=False)
    records = []
    for _, row in annotation.iterrows():
        categories = DIMENSION_CATEGORIES[row['Dimension']]
        lookup = {c.lower(): c for c in categories}
        selected = [s.strip().lower() for s in str(row['Selected Categories']).split(';') if s.strip()]
        unknown = [s for s in selected if s not in lookup]
        if unknown:
            raise ValueError(f'PMID {row["Pmid"]}, {row["Dimension"]}: unknown categories {unknown}')
        records += [{'Pmid': row['Pmid'], 'Dimension': row['Dimension'], 'Category': c,
                     'Value': c.lower() in selected} for c in categories]
    return pd.DataFrame(records)


human_df = load_human_annotation(human_file)
if not human_df['Value'].any():
    print('Warning: no categories selected yet, fill in the annotation first')

human_runs = {name: {**run, 'df': run['df'][run['df']['Pmid'].isin(human_pmids)]}
              for name, run in results.items()}
kappa_table(human_runs, human_df, 'Me').round(2)

In [ ]:
disagreements(human_runs['Jev gated']['df'], human_df, 'Jev', 'Me').query('Total > 0')

In [ ]:
# Side by side per abstract and dimension: which categories each rater selected
def selected(df, name):
    return (df[df['Value']].groupby(['Pmid', 'Dimension'], sort=False)['Category']
            .agg('; '.join).rename(name))


index = pd.MultiIndex.from_product([human_pmids, list(DIMENSION_CATEGORIES)], names=['Pmid', 'Dimension'])
pd.concat([selected(human_df, 'Me')] + [selected(run['df'], name) for name, run in human_runs.items()],
          axis=1).reindex(index).fillna('')